In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

## Baseline Models

All baseline models are trained on GUIDE-seq and evaluated on both GUIDE-seq (in-distribution) 
and CHANGE-seq (out-of-distribution) to demonstrate the cross-dataset distribution shift.

In [4]:
# Load preprocessed one-hot encoded arrays from disk
# These were generated in preprocessing.ipynb - no need to rerun that notebook

DATA_DIR = 'data/processed/'

X_guide_train = np.load(DATA_DIR + 'X_guide_train.npy')
X_guide_test = np.load(DATA_DIR + 'X_guide_test.npy')
X_change_train = np.load(DATA_DIR + 'X_change_train.npy')
X_change_test = np.load(DATA_DIR + 'X_change_test.npy')

# Labels
y_guide_train = np.load(DATA_DIR + 'y_guide_train.npy')
y_guide_test = np.load(DATA_DIR + 'y_guide_test.npy')
y_change_train = np.load(DATA_DIR + 'y_change_train.npy')
y_change_test = np.load(DATA_DIR + 'y_change_test.npy')

# Sanity check
print(f'X_guide_train: {X_guide_train.shape}, y_guide_train: {y_guide_train.shape}')
print(f'X_change_train: {X_change_train.shape}, y_change_train: {y_change_train.shape}')

X_guide_train: (1229372, 184), y_guide_train: (1229372,)
X_change_train: (2352636, 184), y_change_train: (2352636,)


### Logistic Regression

In [3]:
# Train on GUIDE-seq, evaluate both in-distribution and OOD
# Using class_weight='balanced' to handle the severe class imbalance (~99.9% negatives)
lr_guide = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=2000)
lr_guide.fit(X_guide_train, y_guide_train)

# In-distribution evaluation
y_guide_proba = lr_guide.predict_proba(X_guide_test)[:, 1]
print('GUIDE-seq (in-distribution):')
print(f'  AUROC: {roc_auc_score(y_guide_test, y_guide_proba):.4f}')
print(f'  AUPRC: {average_precision_score(y_guide_test, y_guide_proba):.4f}')

# OOD evaluation - model trained on GUIDE-seq, tested on CHANGE-seq
y_change_proba_lr = lr_guide.predict_proba(X_change_test)[:, 1]
print('CHANGE-seq (out-of-distribution):')
print(f'  AUROC: {roc_auc_score(y_change_test, y_change_proba_lr):.4f}')
print(f'  AUPRC: {average_precision_score(y_change_test, y_change_proba_lr):.4f}')

GUIDE-seq (in-distribution):
  AUROC: 0.6401
  AUPRC: 0.0006
CHANGE-seq (out-of-distribution):
  AUROC: 0.8575
  AUPRC: 0.0826


### Random Forest

In [5]:
# Random forest with class weighting 
# n_estimators=100 is a reasonable default, may be slow on 1.2M rows
rf_guide = RandomForestClassifier(class_weight='balanced', n_estimators=100, 
                                   random_state=2000, n_jobs=-1)
rf_guide.fit(X_guide_train, y_guide_train)

# In-distribution evaluation
y_guide_proba_rf = rf_guide.predict_proba(X_guide_test)[:, 1]
print('GUIDE-seq (in-distribution):')
print(f'  AUROC: {roc_auc_score(y_guide_test, y_guide_proba_rf):.4f}')
print(f'  AUPRC: {average_precision_score(y_guide_test, y_guide_proba_rf):.4f}')

# OOD evaluation
y_change_proba_rf = rf_guide.predict_proba(X_change_test)[:, 1]
print('CHANGE-seq (out-of-distribution):')
print(f'  AUROC: {roc_auc_score(y_change_test, y_change_proba_rf):.4f}')
print(f'  AUPRC: {average_precision_score(y_change_test, y_change_proba_rf):.4f}')

GUIDE-seq (in-distribution):
  AUROC: 0.5484
  AUPRC: 0.0010
CHANGE-seq (out-of-distribution):
  AUROC: 0.5581
  AUPRC: 0.0466
